# Validate the silver tables

A separate pipeline, the job `validate_silver`. It **only reads** the bronze and silver tables. It checks that the tables are consistent, reports how much work is left, and **fails** if a rule that must always hold is broken. Run it after the ingest job, or on a schedule.

It is kept apart from the ingest job on purpose: a validation failure should not mark an ingest run as failed or block it, and the checks can be run on their own at any time, for any schema (the `schema` widget).

| Check | Kind | Meaning |
| --- | --- | --- |
| Every bronze PDF has a record | **must hold** | Each PDF is in `silver_parsed` or in `silver_log`. A PDF with no record would be skipped silently |
| No duplicate pages | **must hold** | `silver_pages` has one row per (`path`, `page_no`) |
| No empty page | **must hold** | A stored page has text |
| No page of an unknown PDF | **must hold** | Every stored page belongs to a PDF that is parsed, or that is waiting for a parse retry (in `silver_log`). Pages are kept for the PDFs waiting for a retry, so they are not converted twice |
| No page beyond the document | **must hold** | A PDF has no more converted pages than pages with content |
| Complete documents | information | PDFs whose pages are all converted (`silver_documents`) |
| Documents still converting | information | Parsed PDFs with missing pages. Normal until the conversion has run |
| Open failures | information | `silver_failures` by stage and whether a retry makes sense |

"Must hold" checks are about the integrity of the tables, so a failure means a bug or a damaged table. "Information" checks show progress and normal leftovers, and never fail the job.

In [ ]:
import sys
sys.path.insert(0, "../silver")  # the shared table names
from tables import Names

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
n = Names(dbutils.widgets.get("catalog"), dbutils.widgets.get("schema"))


def scalar(sql):
    return spark.sql(sql).first()[0] or 0


# (name, number found, must be zero?)
checks = [
    ("PDFs in bronze with no record in silver_parsed or silver_log", scalar(f"""
        SELECT count(*) FROM {n.bronze} b
        WHERE b.path NOT IN (SELECT path FROM {n.parsed}) AND b.path NOT IN (SELECT path FROM {n.log} WHERE stage = 'parse')"""), True),
    ("duplicate pages in silver_pages", scalar(f"""
        SELECT count(*) FROM (SELECT path, page_no FROM {n.pages} GROUP BY path, page_no HAVING count(*) > 1)"""), True),
    ("empty pages in silver_pages", scalar(f"SELECT count(*) FROM {n.pages} WHERE length(trim(markdown)) = 0"), True),
    ("pages of a PDF that is neither parsed nor in silver_log", scalar(f"""
        SELECT count(*) FROM {n.pages}
        WHERE path NOT IN (SELECT path FROM {n.parsed}) AND path NOT IN (SELECT path FROM {n.log} WHERE stage = 'parse')"""), True),
    ("PDFs with more converted pages than pages with content", scalar(f"""
        SELECT count(*) FROM {n.parsed} p JOIN (SELECT path, count(*) AS n FROM {n.pages} GROUP BY path) g ON g.path = p.path
        WHERE g.n > p.pages"""), True),
]
info = {
    "PDFs in bronze": scalar(f"SELECT count(*) FROM {n.bronze}"),
    "PDFs parsed (silver_parsed)": scalar(f"SELECT count(*) FROM {n.parsed}"),
    "pages converted (silver_pages)": scalar(f"SELECT count(*) FROM {n.pages}"),
    "complete documents (silver_documents)": scalar(f"SELECT count(*) FROM {n.documents}"),
    "pages kept for PDFs waiting for a parse retry": scalar(f"""
        SELECT count(*) FROM {n.pages} WHERE path NOT IN (SELECT path FROM {n.parsed})"""),
    "parsed PDFs still missing pages": scalar(f"""
        SELECT count(*) FROM {n.parsed} p LEFT JOIN (SELECT path, count(*) AS n FROM {n.pages} GROUP BY path) g ON g.path = p.path
        WHERE coalesce(g.n, 0) < p.pages"""),
    "pages still to convert": scalar(f"""
        SELECT sum(p.pages - coalesce(g.n, 0)) FROM {n.parsed} p LEFT JOIN (SELECT path, count(*) AS n FROM {n.pages} GROUP BY path) g ON g.path = p.path"""),
}

print("MUST HOLD")
for name, found, _ in checks:
    print(f"  {'OK  ' if found == 0 else 'FAIL'} {name}: {found}")
print("\nINFORMATION")
for name, value in info.items():
    print(f"  {value:>8}  {name}")

## Open failures

What is still open in `silver_failures`: PDFs not parsed and pages not converted. A retryable failure is picked up by the next ingest run. A failure that is not retryable (a PDF over 500 pages, a failed conversion) needs a different fix.

In [ ]:
display(spark.table(n.failures).groupBy("stage", "retryable").count().orderBy("stage", "retryable"))
display(spark.table(n.failures).orderBy("stage", "path").limit(100))

## Result

Fails the task if a "must hold" check found a problem, so the job run shows red.

In [ ]:
broken = [f"{name}: {found}" for name, found, must_be_zero in checks if must_be_zero and found != 0]
assert not broken, "Validation failed:\n  " + "\n  ".join(broken)
print("All checks that must hold passed.")